# 00b — GEO conversion: human embryo weeks 3–4, gene names matched to the trunk morph data

**Feeds:** `01_preprocessing.ipynb`, which reads the three files this notebook writes

**Needs:** `sample_284.adata` from `00a_geo_trunk_morph.ipynb`, for its gene IDs and names

Ported from the authors' notebook `day6_PDMS+glass+limbbud/embryo-w3-4+org_SZ3.12-Copy1.ipynb`, cells 2,
4, 5, 6 and 7 (the notebook's cell list, counted from 0). Cell 2 imports; cells 4 and 5 read Zeng et al.'s
human embryo data and keep libraries W3-1, W4-1 and W4-2; cell 6 reads the trunk morph object; cell 7 renames
each embryo gene to the trunk morph gene name that has the same `gene_ids`, lists the genes found in only
one dataset, and writes the outputs. In the original notebook cells 4, 5 and 7 are commented out, and the
next cell reads their outputs back from disk. They are uncommented here by removing the leading `# ` from
each line.

**Input** — GEO series GSE155121, `GSE155121_human_data_raw.h5ad.gz`, decompressed (`gunzip`; about 10 GB)
to `data/GSE155121/GSE155121_human_data_raw.h5ad` (or `$GEO_DATA_DIR/GSE155121/`). A backed read needs the
uncompressed file.

**Outputs** — written to `$SCRNASEQ_INPUT_ROOT` (default `data/scrnaseq_inputs/`), where
`01_preprocessing` reads them:
- `human_embryo/GSE155121/GSE155121_human_data_w3-4_genematch.h5ad`
- `embryo_only_genes.pkl`
- `morph_only_genes.pkl`

**Changes from the original notebook**
1. The raw file is opened backed (read-only, left on disk), the `week_stage` libraries W3-1, W4-1 and W4-2
   are selected, and only that subset is loaded into memory (`.to_memory()`). The original loaded all
   463,304 cells first: 1.25 billion non-zero values, about 10 GB in memory.
2. The trunk morph gene list is written as `morph_only_genes.pkl`, the name `01_preprocessing` reads. The
   original wrote `organoid_only_genes.pkl`. Variable names inside the code are unchanged.
3. Paths: the first code cell was added to locate the GEO file, `sample_284.adata` and the output
   directory. The original used paths relative to its working directory.

No other line of code was changed.

In [ ]:
from pathlib import Path
import os
import sys

# Paths. Added for this repository: the original notebook read and wrote in its working directory.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "trunk_morph_ref").is_dir())
# GEO downloads: GEO_DATA_DIR/GSE155121/ (default data/GSE155121/).
GEO_DATA_DIR = Path(os.environ.get("GEO_DATA_DIR", REPO / "data"))
# 01_preprocessing reads what this notebook writes from $SCRNASEQ_INPUT_ROOT (default data/scrnaseq_inputs/).
sys.path.insert(0, str(REPO))
from src.trunk_morph_ref.paths import scrnaseq_input_root
SCRNASEQ_INPUT_ROOT = scrnaseq_input_root(REPO)
(SCRNASEQ_INPUT_ROOT / "human_embryo" / "GSE155121").mkdir(parents=True, exist_ok=True)
print("reading GEO files from", GEO_DATA_DIR / "GSE155121")
print("reading sample_284.adata from and writing to", SCRNASEQ_INPUT_ROOT)


In [ ]:
import pickle
import random

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy
import seaborn as sns
from scipy.sparse import coo_matrix, csr_matrix, load_npz, save_npz

In [ ]:
# Read human embryo data
# Zeng et al, Cell Stem Cell 2023, https://doi.org/10.1016/j.stem.2023.04.016
adata_embryo_raw = sc.read_h5ad(GEO_DATA_DIR / "GSE155121" / "GSE155121_human_data_raw.h5ad", backed="r")

# Print number of cells per tissue sample (week_stage)
adata_embryo_raw.obs.value_counts()

In [ ]:
# We analyze week 3 and week 4 embryo data together
adata_embryo_raw = adata_embryo_raw[adata_embryo_raw.obs.week_stage.isin(["W3-1", "W4-1", "W4-2"])].to_memory()
adata_embryo_raw.obs["source"] = adata_embryo_raw.obs[
    "week_stage"
].cat.rename_categories({"W3-1": "W3-1 Embryo", "W4-1": "W4-1 Embryo", "W4-2": "W4-2 Embryo"})

In [ ]:
# Read organoid data
adata_organoid_raw = sc.read_h5ad(SCRNASEQ_INPUT_ROOT / "sample_284.adata")

# Only analyze day 6 "No Bead" and "BMP4 Bead" organoids (cells with detected guides)
adata_organoid_raw = adata_organoid_raw[
    adata_organoid_raw.obs.guide.isin(["HES7", "ETV4"])
]
adata_organoid_raw.obs["source"] = adata_organoid_raw.obs[
    "guide"
].cat.rename_categories({"ETV4": "No Bead Organoid", "HES7": "BMP4 Bead Organoid"})

# Print number of cells per organoid sample
adata_organoid_raw.obs.source.value_counts()

In [ ]:
# Homogenize gene names between datasets using gene_ids

# I have confirmed that all gene_ids are unique
# embryo: 32738 gene_ids
# organoid: 36601 gene_ids
embryo_only_genes = []
organoid_only_genes = []

# Change embryo gene names to organoid gene names
for i in range(len(adata_embryo_raw.var)):
    g = adata_embryo_raw.var.index[i]
    id = adata_embryo_raw.var.iloc[i]["gene_ids"]
    try:
        g_o = adata_organoid_raw.var.index[adata_organoid_raw.var.gene_ids==id][0]
        adata_embryo_raw.var.rename(index={g: g_o}, inplace=True)
    except:
        embryo_only_genes.append(g)

for i in range(len(adata_organoid_raw.var)):
    g = adata_organoid_raw.var.index[i]
    id = adata_organoid_raw.var.iloc[i]["gene_ids"]
    if id not in adata_embryo_raw.var["gene_ids"].values:
        organoid_only_genes.append(g)

adata_embryo_raw.write_h5ad(
    filename=SCRNASEQ_INPUT_ROOT / "human_embryo" / "GSE155121" / "GSE155121_human_data_w3-4_genematch.h5ad"
)
with open(SCRNASEQ_INPUT_ROOT / "embryo_only_genes.pkl", "wb") as f:
    pickle.dump(embryo_only_genes, f)
with open(SCRNASEQ_INPUT_ROOT / "morph_only_genes.pkl", "wb") as f:
    pickle.dump(organoid_only_genes, f)